# 📰 Fake News Detection using Graph ML
## Notebook 4: Visualization & Analysis

**What we visualize:**
1. t-SNE plot of GNN embeddings (Real vs Fake clusters)
2. Confusion matrices for all 3 models
3. ROC curves comparison
4. GAT attention weight heatmap
5. Final knowledge graph with predicted labels

In [ ]:
!pip install torch-geometric scikit-learn matplotlib seaborn -q
print('✅ Ready!')

In [ ]:
import torch, pickle, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import torch.nn.functional as F

from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, roc_curve, auc
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
import torch.nn as nn

os.makedirs('results', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print(f'✅ Libraries loaded | Device: {device}')

## Step 1 — Load Data & Saved Models

In [ ]:
# Load graph and metadata
graph_data = torch.load('data/graph_data.pt').to(device)
with open('data/meta.pkl', 'rb') as f:
    meta = pickle.load(f)
with open('data/results.pkl', 'rb') as f:
    saved_results = pickle.load(f)

train_df = pd.read_csv('data/train.csv')
val_df   = pd.read_csv('data/val.csv')
test_df  = pd.read_csv('data/test.csv')
df       = pd.concat([train_df, val_df, test_df], ignore_index=True)

IN_DIM = graph_data.num_node_features
print(f'✅ Data loaded | Feature dim: {IN_DIM}')

In [ ]:
# Re-define and reload models
class GCN(nn.Module):
    def __init__(self, in_dim, hidden=128, out_dim=2, dropout=0.5):
        super().__init__()
        self.conv1   = GCNConv(in_dim, hidden)
        self.conv2   = GCNConv(hidden, hidden // 2)
        self.linear  = nn.Linear(hidden // 2, out_dim)
        self.dropout = dropout
    def forward(self, data):
        x, ei = data.x, data.edge_index
        x = F.relu(self.conv1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, ei))
        x = self.linear(x)
        return F.log_softmax(x, dim=1), x

class GAT(nn.Module):
    def __init__(self, in_dim, hidden=64, out_dim=2, heads=4, dropout=0.5):
        super().__init__()
        self.conv1  = GATConv(in_dim, hidden, heads=heads, dropout=dropout)
        self.conv2  = GATConv(hidden * heads, hidden, heads=1, concat=False, dropout=dropout)
        self.linear = nn.Linear(hidden, out_dim)
        self.dropout = dropout
    def forward(self, data, return_attention=False):
        x, ei = data.x, data.edge_index
        x = F.dropout(x, p=self.dropout, training=self.training)
        if return_attention:
            x, (edge_idx, alpha1) = self.conv1(x, ei, return_attention_weights=True)
        else:
            x = self.conv1(x, ei)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv2(x, ei))
        x = self.linear(x)
        if return_attention:
            return F.log_softmax(x, dim=1), x, alpha1.mean(dim=1), edge_idx
        return F.log_softmax(x, dim=1), x

class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden=128, out_dim=2, dropout=0.5):
        super().__init__()
        self.conv1  = SAGEConv(in_dim, hidden)
        self.conv2  = SAGEConv(hidden, hidden // 2)
        self.linear = nn.Linear(hidden // 2, out_dim)
        self.dropout = dropout
    def forward(self, data):
        x, ei = data.x, data.edge_index
        x = F.relu(self.conv1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, ei))
        x = self.linear(x)
        return F.log_softmax(x, dim=1), x

gcn_model  = GCN(IN_DIM).to(device)
gat_model  = GAT(IN_DIM).to(device)
sage_model = GraphSAGE(IN_DIM).to(device)

gcn_model.load_state_dict(torch.load('models/gcn.pt',  map_location=device))
gat_model.load_state_dict(torch.load('models/gat.pt',  map_location=device))
sage_model.load_state_dict(torch.load('models/sage.pt', map_location=device))

for m in [gcn_model, gat_model, sage_model]:
    m.eval()

print('✅ All models loaded and in eval mode')

## Step 2 — t-SNE Embedding Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models_list = [
    (gcn_model,  'GCN',        '#3498db', '#e74c3c'),
    (gat_model,  'GAT',        '#3498db', '#e74c3c'),
    (sage_model, 'GraphSAGE',  '#3498db', '#e74c3c'),
]

n_art = meta['n_articles']
y_all = graph_data.y[:n_art].cpu().numpy()

for ax, (model, name, c_real, c_fake) in zip(axes, models_list):
    with torch.no_grad():
        _, emb = model(graph_data)
    emb_np = emb[:n_art].cpu().numpy()

    # t-SNE on a random sample of 2000 nodes for speed
    sample_idx = np.random.choice(n_art, size=min(2000, n_art), replace=False)
    emb_sample = emb_np[sample_idx]
    y_sample   = y_all[sample_idx]

    tsne  = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=500)
    z     = tsne.fit_transform(emb_sample)

    real_mask = y_sample == 0
    fake_mask = y_sample == 1

    ax.scatter(z[real_mask, 0], z[real_mask, 1], c=c_real, s=8, alpha=0.6, label='Real')
    ax.scatter(z[fake_mask, 0], z[fake_mask, 1], c=c_fake, s=8, alpha=0.6, label='Fake')
    ax.set_title(f't-SNE — {name}', fontsize=13)
    ax.legend(markerscale=3)
    ax.axis('off')

plt.suptitle('GNN Node Embeddings (t-SNE projection)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('results/tsne_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 t-SNE plot saved!')

## Step 3 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (model, name) in zip(axes, [
    (gcn_model, 'GCN'), (gat_model, 'GAT'), (sage_model, 'GraphSAGE')
]):
    with torch.no_grad():
        out, _ = model(graph_data)
    pred   = out.argmax(dim=1)
    y_true = graph_data.y[graph_data.test_mask].cpu().numpy()
    y_pred = pred[graph_data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap='Blues',
                xticklabels=['Real', 'Fake'],
                yticklabels=['Real', 'Fake'])
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('results/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Confusion matrices saved!')

## Step 4 — ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))

colors = {'GCN': '#3498db', 'GAT': '#e74c3c', 'GraphSAGE': '#2ecc71'}

for model, name in [(gcn_model, 'GCN'), (gat_model, 'GAT'), (sage_model, 'GraphSAGE')]:
    with torch.no_grad():
        out, _ = model(graph_data)
    probs  = torch.exp(out)[:, 1]  # probability of fake
    y_true = graph_data.y[graph_data.test_mask].cpu().numpy()
    y_prob = probs[graph_data.test_mask].cpu().numpy()

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc     = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[name], lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
plt.xlim([0, 1]); plt.ylim([0, 1.02])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All GNN Models')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('results/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 ROC curves saved!')

## Step 5 — GAT Attention Weight Heatmap

In [ ]:
gat_model.eval()
with torch.no_grad():
    _, _, attn_weights, edge_idx = gat_model(graph_data, return_attention=True)

# Visualize attention for a small subgraph (first 30 article nodes)
attn_np   = attn_weights.cpu().numpy()
edge_src  = edge_idx[0].cpu().numpy()
edge_dst  = edge_idx[1].cpu().numpy()

# Build 30x30 attention matrix for article-article attention (through shared speakers)
N = 30
attn_matrix = np.zeros((N, N))

for i, (s, d, w) in enumerate(zip(edge_src, edge_dst, attn_np)):
    if s < N and d < N:
        attn_matrix[s, d] = w

plt.figure(figsize=(10, 8))
mask = attn_matrix == 0
sns.heatmap(attn_matrix,
            mask=mask,
            cmap='YlOrRd',
            linewidths=0.3,
            xticklabels=[f'N{i}' for i in range(N)],
            yticklabels=[f'N{i}' for i in range(N)])
plt.title('GAT Attention Weights (first 30 nodes)')
plt.xlabel('Target node')
plt.ylabel('Source node')
plt.tight_layout()
plt.savefig('results/gat_attention.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 GAT attention heatmap saved!')

## Step 6 — Final Graph with Predicted Labels

In [ ]:
# Use GAT (best attention model) predictions on a 60-node subgraph
gat_model.eval()
with torch.no_grad():
    out, _ = gat_model(graph_data)
preds_all = out.argmax(dim=1).cpu().numpy()
truth_all = graph_data.y.cpu().numpy()

# Build sample subgraph
SAMPLE = 60
G = nx.Graph()
node_colors, node_sizes, node_shapes = [], [], []

for i in range(SAMPLE):
    pred  = preds_all[i]
    truth = truth_all[i]
    correct = (pred == truth)
    # Color: correct=green/pink, wrong=orange
    if correct:
        color = '#2ecc71' if pred == 0 else '#e74c3c'
    else:
        color = '#f39c12'   # wrong prediction
    G.add_node(i, node_type='article')
    node_colors.append(color)
    node_sizes.append(150)

# Add speaker/subject nodes
from sklearn.preprocessing import LabelEncoder
se = LabelEncoder().fit(df['speaker'])
su = LabelEncoder().fit(df['subject'])
df['sid'] = se.transform(df['speaker'])
df['subid'] = su.transform(df['subject'])

n_art = meta['n_articles']
n_spk = meta['n_speakers']

for i in range(SAMPLE):
    spk_key = f"s_{df.iloc[i]['sid']}"
    sub_key = f"t_{df.iloc[i]['subid']}"
    if spk_key not in G:
        G.add_node(spk_key, node_type='speaker')
        node_colors.append('#3498db')
        node_sizes.append(60)
    if sub_key not in G:
        G.add_node(sub_key, node_type='subject')
        node_colors.append('#9b59b6')
        node_sizes.append(60)
    G.add_edge(i, spk_key)
    G.add_edge(i, sub_key)

plt.figure(figsize=(16, 12))
pos = nx.spring_layout(G, k=0.5, seed=42)
nx.draw_networkx(G, pos,
                 node_color=node_colors,
                 node_size=node_sizes,
                 labels={n: '' for n in G.nodes()},
                 edge_color='#dddddd',
                 width=0.5, alpha=0.9)

legend_handles = [
    mpatches.Patch(color='#2ecc71', label='Correctly predicted Real'),
    mpatches.Patch(color='#e74c3c', label='Correctly predicted Fake'),
    mpatches.Patch(color='#f39c12', label='Wrong prediction'),
    mpatches.Patch(color='#3498db', label='Speaker node'),
    mpatches.Patch(color='#9b59b6', label='Subject node'),
]
plt.legend(handles=legend_handles, loc='upper left', fontsize=11)
plt.title('Knowledge Graph — GAT Predictions\n(green=real correct, red=fake correct, orange=wrong)', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.savefig('results/predicted_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Prediction graph saved to results/')

## 🎉 Project Complete!

### Summary of all outputs
| File | Description |
|------|-------------|
| `results/tsne_embeddings.png` | 2D embedding clusters for all 3 models |
| `results/confusion_matrices.png` | Confusion matrices |
| `results/roc_curves.png` | ROC curves comparison |
| `results/gat_attention.png` | GAT attention heatmap |
| `results/predicted_graph.png` | Final knowledge graph with predictions |
| `results/metrics.csv` | Accuracy, F1, AUC table |

### Next steps for your report
- Compare GCN vs GAT vs GraphSAGE in a table
- Explain why GAT performs best (attention weights)
- Discuss the graph structure's role in propagating credibility signals